In [1]:
# import libraries

import numpy as np
import pandas as pd
import geopandas as gpd

import leafmap.maplibregl as leafmap

import os

Load files

In [2]:
# Load total pop
total_pop_folder = "Population Data"
pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_age_breakdown.parquet"))

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

In [3]:
# Create total, primary and secondary population gdfs
total_pop_gdf = pop_gdf.copy()[["id","total_pop","x","y","bandar_luarbandar","geometry"]]

# Round numbders
total_pop_gdf["total_pop"] = total_pop_gdf["total_pop"].round(0).astype(int)

# Rename columns
total_pop_gdf = total_pop_gdf.rename(columns={"id":"pop_id","total_pop":"total_population"})

Interactive map

In [4]:
output_folder = "Interactive Maps"

In [5]:
# -----------------------------
# 1. Define output folders
# -----------------------------

base_output_folder = output_folder   # your original output folder
assets_folder = os.path.join(base_output_folder, "assets")
os.makedirs(assets_folder, exist_ok=True)

# HTML output file
output_path = os.path.join(
    base_output_folder,
    "swk_bandar_luarbandar_pop_interactive_map.html"
)

# -----------------------------
# 2. Build bandar vs luar bandar gdfs
# -----------------------------

gdf_bandar = total_pop_gdf[total_pop_gdf["bandar_luarbandar"] == "Bandar"].copy()
gdf_luarbandar = total_pop_gdf[total_pop_gdf["bandar_luarbandar"] == "Luar Bandar"].copy()

# -----------------------------
# 3. Build the interactive map
# -----------------------------

x = (total_pop_gdf["x"].max() + total_pop_gdf["x"].min()) / 2
y = (total_pop_gdf["y"].max() + total_pop_gdf["y"].min()) / 2

m = leafmap.Map(height="600px", center=[x, y], use_message_queue=True, style="street")

# Basemap
m.add_basemap(
    "Esri.WorldImagery",
    before_id=m.first_symbol_layer_id,
    visible=True
)

m.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery (Plain)",
    attribution="Esri World Imagery",
    visible=False
)


# Data layers
m.add_gdf(
    gdf=gdf_bandar,
    name="Bandar Population",
    layer_type="circle",
    paint={"circle-radius": 3, "circle-color": "#0077FF"},
    visible=True
)

m.add_gdf(
    gdf=gdf_luarbandar,
    name="Luar Bandar Population",
    layer_type="circle",
    paint={"circle-radius": 3, "circle-color": "#000000"},
    visible=True
)


m.add_gdf(
    swk_districts_gdf,
    name="District Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=True
)

m.add_gdf(
    swk_parlimen_gdf,
    name="Parlimen Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

m.add_gdf(
    swk_dun_gdf,
    name="DUN Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

# -----------------------------
# 4. Add map title / sources
# -----------------------------

m.add_text(
    "Population broken down by Bandar vs Luar Bandar",
    position="top-left",
    font_size=12
)

# -----------------------------
# 5. Add legend
# -----------------------------

colour_map = {
    "Bandar": "#0077FF",  
    "Luar Bandar": "#000000",  
}

m.add_legend(
    title = (
    "Bandar vs Luar Bandar"),
    legend_dict = colour_map,
    position = "bottom-right"
)


# -----------------------------
# 6. Layer control
# -----------------------------

m.add_layer_control(
    layer_ids=[
        "Bandar Population",
        "Luar Bandar Population",
        "District Boundaries",
        "Parlimen Boundaries",
        "DUN Boundaries",
        "Esri.WorldImagery",
        "Esri.WorldImagery (Plain)"
    ],
    position="top-left"
)

# -----------------------------
# 7. Export HTML
# -----------------------------

m.to_html(
    output=output_path,
    title="Population broken down by Bandar vs Luar Bandar (Estimated)",
    overwrite=True
)

m

Html(children=[<leafmap.maplibregl.Map object at 0x0000014C14B3FAC0>, Card(children=[Btn(children=[Icon(childr…